# Model E — Intelligent Hint Generation

##  1. Import Libraries

In [ ]:
import os
import re
import warnings
from tqdm import tqdm 
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

##  2. Load Raw Dataset

In [ ]:

train_df = pd.read_csv("../data/raw/train.csv")

print(f" Dataset loaded — {len(train_df):,} samples ready")

 Dataset loaded — 87,866 samples ready


##  3. Sentence Tokeniser

In [3]:
def split_into_sentences(article):
    """
    Split an article string into a list of individual sentences.
    Uses punctuation boundaries (. ! ?) as split points.
    Empty strings are filtered out after stripping whitespace.
    """
    sentences = re.split(r"(?<=[.!?])\s+", str(article))
    return [s.strip() for s in sentences if s.strip()]


## 4. Core Function: Generate Graduated Hints


In [4]:
def generate_hints(article, question, top_n=3):
    """
    Generate three graduated hints for a reading comprehension question.

    Parameters
    ----------
    article  : str — full article text (raw, unprocessed)
    question : str — the comprehension question
    top_n    : int — number of candidate sentences to retrieve (default 3)

    Returns
    -------
    list[str] — [general_hint, specific_hint, explicit_hint]
    """
    sentences = split_into_sentences(article)

    # Safety fallback: article too short to rank meaningfully
    if len(sentences) < 3:
        return [
            "Read the passage carefully from the beginning.",
            "Look for keywords related to the question.",
            "Focus on the main topic discussed in the passage.",
        ]

    try:
        # Build TF-IDF matrix: row 0 = question, rows 1+ = article sentences
        corpus = [str(question)] + sentences
        tfidf_matrix = TfidfVectorizer(stop_words="english").fit_transform(corpus)

        # Score every sentence by its similarity to the question
        similarities = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
        ranked_indices = np.argsort(similarities)[::-1]

        # Take top-N then reverse: index 0 = least relevant, index 2 = most relevant
        top_sentences = [sentences[i] for i in ranked_indices[:top_n]][::-1]

        # Wrap each sentence in a prefix that signals its specificity level
        hints = [
            "Focus on this section: "  + top_sentences[0][:120],  # vague
            "This sentence may help: " + top_sentences[1][:150],  # direct
            "Most relevant clue: "     + top_sentences[2],        # explicit (full)
        ]
        return hints

    except Exception:
        return [
            "Re-read the passage carefully.",
            "Look for related keywords.",
            "Focus on the concluding sentences.",
        ]


##  5. Run Hint Generation Pipeline


In [ ]:

sample_df = train_df 
results = []

print(f"Generating hints for all {len(sample_df):,} questions. This will take a while...")

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    hints = generate_hints(row["article"], row["question"])
    
    results.append({
        "Question":        row["question"],
        "Hint_1_General":  hints[0],
        "Hint_2_Specific": hints[1],
        "Hint_3_Explicit": hints[2],
    })

results_df = pd.DataFrame(results)
print(f"\nHints generated for {len(results_df):,} questions")
results_df.head()

Generating hints for all 87,866 questions. This will take a while...


  0%|          | 0/87866 [00:00<?, ?it/s]

100%|██████████| 87866/87866 [03:20<00:00, 437.28it/s]



Hints generated for 87,866 questions


,Question,Hint_1_General,Hint_2_Specific,Hint_3_Explicit
0,Before the writer came to the high school summ...,Focus on this section: I want you to know that...,This sentence may help: The days went by and t...,Most relevant clue: In the summer between my f...
1,How many times did the writer invite the boy t...,Focus on this section: That evening I was told...,"This sentence may help: The mother said: ""'Tom...","Most relevant clue: I walked towards him, intr..."
2,The bumpkin thought _ .,Focus on this section: I haven't brought my wi...,This sentence may help: A bumpkin went to a ...,Most relevant clue: The bumpkin opened his eye...
3,The room he saw was perhaps _ .,Focus on this section: I haven't brought my wi...,This sentence may help: After a while the door...,Most relevant clue: He went into an office bui...
4,We can see that the bumpkin had no knowledge ...,Focus on this section: I haven't brought my wi...,This sentence may help: A bumpkin went to a ...,Most relevant clue: The bumpkin opened his eye...


##  6. Evaluate Hint Quality


In [6]:
all_hints = (
    results_df["Hint_1_General"].tolist() +
    results_df["Hint_2_Specific"].tolist() +
    results_df["Hint_3_Explicit"].tolist()
)

unique_count = len(set(all_hints))
avg_length   = np.mean([len(str(h).split()) for h in all_hints])

print(f" Unique hints generated   : {unique_count}")
print(f" Average hint word length : {avg_length:.2f} words")

 Unique hints generated   : 201974
 Average hint word length : 20.66 words


##  7. Save Results

In [7]:
output_path = "../reports/intelligent_hint_results.csv"
os.makedirs("../reports", exist_ok=True)

results_df.to_csv(output_path, index=False)
print(f" Results saved to: {output_path}")

 Results saved to: ../reports/intelligent_hint_results.csv
